In [ ]:
print('Importing libraries...')
import json
import os
import sys
from pathlib import Path
import warnings 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim

from scripts.env import set_deterministic_behaviour
from scripts.dataset import CachedDatasetTempMultilabel
from scripts.refactoring_caching_temporal import cache_validation
from scripts.evidential_metrics import update_model_output_dict, calculate_evidential_metrics, calculate_binary_metrics
from scripts.lora import inject_lora_into_dinov3_qkv
from scripts.model import SequenceEvidentialModel
from scripts.bbkl_loss import total_bb_loss
warnings.filterwarnings("ignore")

In [ ]:
##############################################################################################
##############################################################################################
PWD = Path.cwd()
print(f"PWD: {PWD}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory Allocated: {torch.cuda.memory_allocated(i) / 1024**2:.1f} MB")
        print(f"  Memory Cached:    {torch.cuda.memory_reserved(i) / 1024**2:.1f} MB")
else:
    device = 'cpu'


seed = 0
set_deterministic_behaviour(seed)

annotations_path = PWD / 'config/reformatted_annotations_temporal.json'
dataset_dir = PWD.parent / 'dataset/endoscapes/'
cached_images_path = Path('/media/franek/D0F29AD9F29AC2E0/Users/Franek/Documents/python code/Linux_Cache') / 'cached_images_temporal_multilabel'

# Parameters to create the dataset
force_recache = False

# These are declared in scripts/refactoring_caching.py
# IMAGE_SIZE = (384, 384)
# DATASET_MEAN = (0.454315, 0.290313, 0.299898)
# DATASET_STD = (0.167318, 0.156652, 0.150197)

cache_validation(   dataset_dir,
                    cached_images_path,
                    annotations_path,
                    force_recache = force_recache)

In [ ]:
##############################################################################################
##############################################################################################
# Paths
train_set_path = cached_images_path / 'train'
val_set_path = cached_images_path / 'val'
test_set_path = cached_images_path / 'test'
# Datasets
dataset_train = CachedDatasetTempMultilabel(train_set_path,
                                label_criterion = (None, 'soft'))
dataset_val = CachedDatasetTempMultilabel(  val_set_path,
                                label_criterion = (None, 'soft'))
dataset_test = CachedDatasetTempMultilabel( test_set_path,
                                label_criterion = (None, 'soft'))
# Dataloaders
train_dataloader = DataLoader(  dataset_train,
                                batch_size = 32,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = True)
val_dataloader = DataLoader(    dataset_val,
                                batch_size = 32,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = False)
test_dataloader = DataLoader(   dataset_test,
                                batch_size = 32,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = False)

In [ ]:
##############################################################################################
##############################################################################################
# Init DinoV3 Backbone
REPO_DIR = PWD.parent / 'dinov3'
sys.path.insert(0, str(REPO_DIR))
dinov3_pretarined_weights = './weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'
frozen_encoder = torch.hub.load(    REPO_DIR,
                                    'dinov3_vitb16',
                                    source='local',
                                    weights=dinov3_pretarined_weights)
# Set head to match the pretraining
in_feats = 768
frozen_encoder.head = nn.Sequential(
    nn.Linear(in_feats, in_feats),
    nn.GELU(),
    nn.Linear(in_feats, 3)
)
frozen_encoder.to(device)
####### Freeze the encoder ######
for p in frozen_encoder.parameters():
    p.requires_grad = False
        
## Inject LoRA exactly at attn.qkv (all blocks)
#num_blocks = len(frozen_encoder.blocks)
#target_layers = list(range(num_blocks - 6, num_blocks))
#lora_modules = inject_lora_into_dinov3_qkv(frozen_encoder, r=6, layers=target_layers, verbose=True, alpha = 12)

# Load pretraining checkpoint
pretrained_checkpoint = PWD / 'weights/DinoV3_multilabel_pretraining_no_lora_6enc.pt'
pretrained_model_weights = torch.load(pretrained_checkpoint, map_location=device)
frozen_encoder.load_state_dict(pretrained_model_weights)
frozen_encoder.head = nn.Identity()
####### Freeze the whole encoder (ONLY USE IN PRETRAINED NETWORK) ######
for p in frozen_encoder.parameters():
    p.requires_grad = False
model = SequenceEvidentialModel(frozen_encoder)
model.to(device)
# Separate parameter groups
pool_params = []
head_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith("heads"):
        head_params.append(p)
    elif name.startswith("pool"):
        pool_params.append(p)
optimizer = optim.AdamW(
    [
        {"params": pool_params, "lr": 1e-4},
        {"params": head_params, "lr": 1e-4},
    ],
    weight_decay=1e-2
)
#weights = {'C1': (w_pos, w_neg),
#           'C2': (w_pos, w_neg),
#           'C3': (w_pos, w_neg)}
# w_pos e.g. 1.76
# w_neg e.g. 0.54
#prior_alpha = { 'C1': (prior_a0, prior_a1),
#                'C2': (prior_a0, prior_a1),
#                'C3': (prior_a0, prior_a1)}
# prior_a0 = (1-pi)*v e.g. (1-0.05)*2 = 0.95*2 = 1.9
# prior_a1 = pi*v, e.g. 0.05*2 = 0.1
use_bb_loss_weights = True
if use_bb_loss_weights:
    weights_bb_loss = { 'C1': (3.1985, 0.5926),
                        'C2': (4.4615, 0.5631),
                        'C3': (2.7952, 0.6089)}
else:
    weights_bb_loss = { 'C1': (1, 1),
                        'C2': (1, 1),
                        'C3': (1, 1)}
use_kl = True
custom_prior_alpha = True
if use_kl and custom_prior_alpha:
    nu = 2
    pi_C1 = 0.1563
    pi_C2 = 0.1121
    pi_C3 = 0.1789
    prior_alpha = {     'C1': ((1-pi_C1)*nu, pi_C1*nu),
                        'C2': ((1-pi_C2)*nu, pi_C2*nu),
                        'C3': ((1-pi_C3)*nu, pi_C3*nu)}
else:
    prior_alpha = None

In [ ]:
##############################################################################################
##############################################################################################
EPOCHS = 10
exp_name = f"evid_bbloss_wkl_wprior_6enc_temporal_{seed}"
results_dict = {}
best_bacc_across_epochs = -1.0
best_epoch = 0
for epoch in range(EPOCHS):
    print(f"Epoch: {epoch+1:02}/{EPOCHS:02}")
    print("Training")
    train_loss_sum = 0.0
    len_train_loader = len(train_dataloader)
    train_output_dict = {'C1': {'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                            'C2': {'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                            'C3': {'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                            'labels':           [],
                            'vid_ids':          [],
                            'frame_ids':        []}
    
    model.train()
    for idx, (images, labels, vid_id, frame_id) in enumerate(train_dataloader):
            print(f'\r{idx+1}/{len_train_loader}', end='', flush=True)
            
            images, labels = images.to(device), labels.to(device)
            torch.cuda.synchronize()
            optimizer.zero_grad()
            output = model(images)
            
            train_loss_total = total_bb_loss(output,
                                            labels,
                                            weights_bb_loss,
                                            use_kl = use_kl,
                                            prior_alpha = prior_alpha)
            train_loss_total.backward()
            optimizer.step()
            
            # Populate the output dict with probs, preds, and uncerts per sample
            train_output_dict = update_model_output_dict(output, train_output_dict)
            train_output_dict['labels'].append(labels.detach().cpu())
            train_output_dict['vid_ids'].append(vid_id.detach().cpu())
            train_output_dict['frame_ids'].append(frame_id.detach().cpu())
            train_loss_sum += train_loss_total.item()
    results, train_output_dict = calculate_evidential_metrics(train_output_dict)
    avg_train_loss = train_loss_sum / len_train_loader
    results['loss'] = round(avg_train_loss, 4)
    print(f"\n--- Training Metrics ---")
    print(f"Train Avg Accuracy              {results['avg_accuracy']:.4f}")
    print(f"Train Avg BAcc                  {results['avg_bacc']:.4f}")
    print(f"Train mAP                       {results['mAP']:.4f}")
    print(f"Train Uncert. Pos:              {results['avg_uncert_pos']:.4f}")
    print(f"Train Uncert. Neg:              {results['avg_uncert_neg']:.4f}")
    print(f"Train Loss:                     {results['loss']:.4f}\n")
    print(f"Train C1 Accuracy               {results['accuracy_C1']:.4f}")
    print(f"Train C2 Accuracy               {results['accuracy_C2']:.4f}")
    print(f"Train C3 Accuracy               {results['accuracy_C3']:.4f}\n")
    print(f"Train C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
    print(f"Train C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
    print(f"Train C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
    print(f"Train C1 AP:                    {results['ap_C1']:.4f}")
    print(f"Train C2 AP:                    {results['ap_C2']:.4f}")
    print(f"Train C3 AP:                    {results['ap_C3']:.4f}")
    print(f"------------------------\n")
    results_dict[f"Epoch {epoch+1} Train"] = results
    
    print('Validation')
    val_loss_sum = 0.0
    len_val_loader = len(val_dataloader)
    val_output_dict = {'C1': {'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                        'C2': {'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                        'C3': {'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                        'labels':           [],
                        'vid_ids':          [],
                        'frame_ids':        []}
    model.eval()
    with torch.inference_mode():
            for idx, (images, labels, vid_id, frame_id) in enumerate(val_dataloader):
                    print(f'\r{idx+1}/{len_val_loader}', end='', flush=True)
                    images, labels = images.to(device), labels.to(device)
                    output = model(images)
                    val_loss_total = total_bb_loss(    output,
                                                        labels,
                                                        weights_bb_loss,
                                                        use_kl = use_kl,
                                                        prior_alpha = prior_alpha)
                    val_output_dict = update_model_output_dict(output, val_output_dict)
                    val_output_dict['labels'].append(labels.detach().cpu())
                    val_output_dict['vid_ids'].append(vid_id.detach().cpu())
                    val_output_dict['frame_ids'].append(frame_id.detach().cpu())
                    val_loss_sum += val_loss_total.item()
    results, val_output_dict = calculate_evidential_metrics(val_output_dict)
    avg_val_loss = val_loss_sum / len_val_loader
    results['loss'] = round(avg_val_loss, 4)
    print(f"\n--- Validation Metrics ---")
    print(f"Val Avg Accuracy              {results['avg_accuracy']:.4f}")
    print(f"Val Avg BAcc                  {results['avg_bacc']:.4f}")
    print(f"Val mAP                       {results['mAP']:.4f}")
    print(f"Val Uncert. Pos:              {results['avg_uncert_pos']:.4f}")
    print(f"Val Uncert. Neg:              {results['avg_uncert_neg']:.4f}")
    print(f"Val Loss:                     {results['loss']:.4f}\n")
    print(f"Val C1 Accuracy               {results['accuracy_C1']:.4f}")
    print(f"Val C2 Accuracy               {results['accuracy_C2']:.4f}")
    print(f"Val C3 Accuracy               {results['accuracy_C3']:.4f}\n")
    print(f"Val C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
    print(f"Val C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
    print(f"Val C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
    print(f"Val C1 AP:                    {results['ap_C1']:.4f}")
    print(f"Val C2 AP:                    {results['ap_C2']:.4f}")
    print(f"Val C3 AP:                    {results['ap_C3']:.4f}")
    print(f"------------------------\n")
    results['saved'] = {'C1': {'probs':      val_output_dict['C1']['probs'].tolist(),
                                'preds':     val_output_dict['C1']['preds'].tolist(),
                                'uncerts':   val_output_dict['C1']['uncerts'].tolist()},
                            'C2': {'probs':     val_output_dict['C2']['probs'].tolist(),
                                'preds':     val_output_dict['C2']['preds'].tolist(),
                                'uncerts':   val_output_dict['C2']['uncerts'].tolist()},
                            'C3': {'probs':     val_output_dict['C3']['probs'].tolist(),
                                'preds':     val_output_dict['C3']['preds'].tolist(),
                                'uncerts':   val_output_dict['C3']['uncerts'].tolist()},
                            'labels':           val_output_dict['labels'].tolist(),
                            'vid_ids':          val_output_dict['vid_ids'].tolist(),
                            'frame_ids':        val_output_dict['frame_ids'].tolist()}
    results_dict[f"Epoch {epoch+1} Val"] = results
    # Save results
    with open(PWD / 'results' / f'{exp_name}_results.json', 'w') as file:
            json.dump(results_dict, file, indent=4)
    # Save weights of the best epoch
    if results['avg_bacc'] >= best_bacc_across_epochs:
            best_bacc_across_epochs = results['avg_bacc']
            best_epoch = epoch+1
            print(f"New best result (Epoch {best_epoch}), saving weights...")
            weights_path = Path.cwd() / 'weights'
            checkpoint_dir = os.path.join(weights_path, f'{exp_name}.pt')
            torch.save(model.state_dict(), checkpoint_dir)
    else:
            print('\n')
print(f"Testing @ epoch {best_epoch}")
test_loss_sum = 0.0
len_test_loader = len(test_dataloader)
test_output_dict = { 'C1': {'probs':     [],
                            'preds':     [],
                            'uncerts':   []},
                    'C2': {'probs':     [],
                            'preds':     [],
                            'uncerts':   []},
                    'C3': {'probs':     [],
                            'preds':     [],
                            'uncerts':   []},
                    'labels':           [],
                    'vid_ids':          [],
                    'frame_ids':        []} 
checkpoint = torch.load(checkpoint_dir, map_location=device)
model.load_state_dict(checkpoint)
model.to(device)
model.eval()
with torch.inference_mode():
    for idx, (images, labels, vid_id, frame_id) in enumerate(test_dataloader):
            print(f'\r{idx+1}/{len_test_loader}', end='', flush=True)
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            test_loss_total = total_bb_loss(   output,
                                                labels,
                                                weights_bb_loss,
                                                use_kl = use_kl,
                                                prior_alpha = prior_alpha)
            test_output_dict = update_model_output_dict(output, test_output_dict)
            test_output_dict['labels'].append(labels.detach().cpu())
            test_output_dict['vid_ids'].append(vid_id.detach().cpu())
            test_output_dict['frame_ids'].append(frame_id.detach().cpu())
            test_loss_sum += test_loss_total.item()
results, test_output_dict = calculate_evidential_metrics(test_output_dict)
avg_test_loss = test_loss_sum / len_test_loader
results['loss'] = round(avg_test_loss, 4)
print(f"\n--- Testing Metrics ---")
print(f"Test Avg Accuracy              {results['avg_accuracy']:.4f}")
print(f"Test Avg BAcc                  {results['avg_bacc']:.4f}")
print(f"Test mAP                       {results['mAP']:.4f}")
print(f"Test Uncert. Pos:              {results['avg_uncert_pos']:.4f}")
print(f"Test Uncert. Neg:              {results['avg_uncert_neg']:.4f}")
print(f"Test Loss:                     {results['loss']:.4f}\n")
print(f"Test C1 Accuracy               {results['accuracy_C1']:.4f}")
print(f"Test C2 Accuracy               {results['accuracy_C2']:.4f}")
print(f"Test C3 Accuracy               {results['accuracy_C3']:.4f}\n")
print(f"Test C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
print(f"Test C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
print(f"Test C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
print(f"Test C1 AP:                    {results['ap_C1']:.4f}")
print(f"Test C2 AP:                    {results['ap_C2']:.4f}")
print(f"Test C3 AP:                    {results['ap_C3']:.4f}")
print(f"------------------------\n")
results['saved'] = {'C1': {'probs':      test_output_dict['C1']['probs'].tolist(),
                            'preds':     test_output_dict['C1']['preds'].tolist(),
                            'uncerts':   test_output_dict['C1']['uncerts'].tolist()},
                    'C2': {'probs':     test_output_dict['C2']['probs'].tolist(),
                            'preds':     test_output_dict['C2']['preds'].tolist(),
                            'uncerts':   test_output_dict['C2']['uncerts'].tolist()},
                    'C3': {'probs':     test_output_dict['C3']['probs'].tolist(),
                            'preds':     test_output_dict['C3']['preds'].tolist(),
                            'uncerts':   test_output_dict['C3']['uncerts'].tolist()},
                    'labels':           test_output_dict['labels'].tolist(),
                    'vid_ids':          test_output_dict['vid_ids'].tolist(),
                    'frame_ids':        test_output_dict['frame_ids'].tolist()}
results_dict[f"Epoch {best_epoch} Test"] = results
with open(PWD / 'results' / f'{exp_name}_results.json', 'w') as file:
    json.dump(results_dict, file, indent=4)